# Разбор отбора: программирование (9–10 класс)

Разбор заданий из `отбор программирование v2.pdf`. Для каждой задачи:

1. **интуиция** — что происходит «на пальцах»;
2. **вектор размышления** — как сужать пространство решений;
3. **ключевая идея** — один принцип, из которого следует алгоритм;
4. **материалы** — формулы, определения, приёмы;
5. **решение** — код и ответ.

Файлы рядом с ноутбуком:

- школьники для задачи 2: `data.csv` / `data.xlsx` (1000 строк, колонки `age`, `math`, `club`);
- логи для задачи 3: `задача 17.xlsx` (лист `logs` — это и есть `timestamps.xlsx` из условия).


## Задача 1. Порог аномальности

Преподаватель считает z-оценки баллов и объявляет выбросом тех, у кого $|z_i| > k$. Значение $k$ забыто, но известно, что выбросов ровно $m$. Нужно восстановить порог: при $m = 0$ вывести `inf`, при $\sigma = 0$ — `0.000000`.


### Интуиция

z-оценка говорит, **на сколько стандартных отклонений** балл далёк от среднего. Чем больше $|z|$, тем «аномальнее» студент.

Порог $k$ — линейка: все, кто дальше $k$, — выбросы. Если линейку поднимать, выбросов становится меньше; если опускать — больше. Нас просят поставить линейку так, чтобы над ней оказалось ровно $m$ человек, и взять ту постановку, которую имеет в виду условие вместе с примерами.


### Вектор размышления

1. $\mu$ и $\sigma$ считаются **один раз по всем** $n$ баллам. Делить на $\sigma$ можно только если $\sigma > 0$.
2. В условии $\sigma$ — это отклонение **генеральной совокупности**:
   $$
   \sigma = \sqrt{\frac{1}{n}\sum_i (x_i-\mu)^2},
   $$
   а не выборочное с $n-1$ в знаменателе.
3. Множество $\{|z_i|\}$ — это $n$ неотрицательных чисел. Число выбросов при пороге $k$ равно количеству значений **строго больших** $k$.
4. Особые случаи закрывают концы шкалы:
   - все баллы равны $\Rightarrow$ формально $z_i$ не определены, по условию ответ `0.000000`;
   - $m = 0$ $\Rightarrow$ печатаем `inf` (пример 2). Это согласуется с тем, что в примере 1 печатают **самое большое** $|z|$, а не второе: авторы берут $m$-ю с конца порядковую статистику модулей, а при нуле выбросов такой статистики нет.
5. Пусть $a_1 \ge a_2 \ge \ldots \ge a_n$ — отсортированные $|z_i|$. Чтобы хотя бы $m$ человек имели $|z| > k$, нужно $k < a_m$. Точной верхней гранью таких $k$ является $a_m$. Именно её и просят как ответ (в примере 1 это $z$ студента с 99 баллами).


### Ключевая идея

**Ответ — $m$-я с конца порядковая статистика модуля z-оценки** (то есть $|z|$ у «последнего» из $m$ самых аномальных). Считать $\mu,\sigma,z$ за линейный проход, затем выбрать эту статистику. $n \le 10^5$, поэтому даже сортировка $O(n\log n)$ проходит спокойно; можно взять `np.partition` за среднее $O(n)$.


### Материалы

**z-оценка (стандартизация):**
$$
z_i = \frac{x_i - \mu}{\sigma},\qquad
\mu = \frac{1}{n}\sum x_i,\qquad
\sigma = \sqrt{\frac{1}{n}\sum (x_i-\mu)^2}.
$$
Геометрически это переход к шкале «среднее = 0, разброс = 1».

**Как не запутаться с `std` в NumPy/pandas:** `numpy.std` по умолчанию делит на $n$ (`ddof=0`) — это то, что нужно. `pandas.Series.std` по умолчанию делит на $n-1$ (`ddof=1`) — для этой задачи **нельзя**.

**Порядок статистик:** если нужен $m$-й максимум массива `a`, это `np.sort(a)[-m]` или `np.partition(a, -m)[-m]`.


### Решение


In [ ]:
import numpy as np


def anomaly_threshold(x, m):
    """Минимальный (в смысле условия) порог k для ровно m выбросов."""
    x = np.asarray(x, dtype=float)
    if m == 0:
        return "inf"

    mu = x.mean()
    sigma = np.sqrt(np.mean((x - mu) ** 2))  # генеральная совокупность, деление на n
    if sigma == 0:
        return 0.0

    abs_z = np.abs((x - mu) / sigma)
    return float(np.partition(abs_z, -m)[-m])


def format_k(k):
    return k if isinstance(k, str) else f"{k:.6f}"


# Пример 1
x1 = [55, 62, 58, 71, 65, 60, 59, 63, 68, 57, 61, 64, 66, 60, 62, 59, 63, 67, 58, 99]
print("пример 1:", format_k(anomaly_threshold(x1, 1)))

# Пример 2
print("пример 2:", format_k(anomaly_threshold([10, 20, 30, 40, 50], 0)))

# Пример 3
print("пример 3:", format_k(anomaly_threshold([7, 7, 7, 7], 4)))


Чтение с консоли (как на контесте):


In [ ]:
def solve_from_stdin():
    n, m = map(int, input().split())
    x = list(map(int, input().split()))
    k = anomaly_threshold(x, m)
    print(k if isinstance(k, str) else f"{k:.10f}")


# solve_from_stdin()


Пояснение к примеру 1: средний балл $63.85$, $\sigma \approx 8.96$, единственный явный хвост — $99$, $|z| \approx 3.92$. В PDF грубо написано $3.870$ и $\mu \approx 63.45$ — округление/опечатка в пояснении; формула в условии однозначная, печатать нужно посчитанное $k$.


## Задача 2. Дерево решений

Дана таблица школьников (`age`, `math`, `club`) и дерево глубины $\le 3$. Нужно:

1. сколько строк уходит в **класс 1**;
2. сколько человек в каждом из **шести листьев** слева направо;
3. для какого `club` доля класса 1 **максимальна** (если несколько — все).


### Интуиция

Дерево — это не «магия ML», а вложенные `if`. Каждый школьник падает вниз по рёбрам ДА/НЕТ и оказывается ровно в одном листе. Класс листа — ответ модели. Вопросы задачи — обычные подсчёты: сколько упало в листья с меткой 1, гистограмма по листьям, группировка по кружку.


### Вектор размышления

1. Сначала **закодировать дерево функцией** `(age, math, club) -> (leaf_id, class)`, чтобы не считать руками.
2. Листья нумеруются **слева направо, как на рисунке**, а не в порядке обхода «сначала все ДА».
3. Доля класса 1 по кружку — это `mean(y == 1)` внутри группы `club`, затем `argmax`.
4. Пропусков нет, значения `club` — ровно четыре: `спорт`, `информатика`, `музыка`, `нет`.


### Ключевая идея

**Явно выписать правила всех шести листьев**, прогнать таблицу векторно (маски pandas/numpy), не обучать `sklearn.DecisionTree` — дерево уже дано.


### Материалы

Дерево из условия:


![Дерево решений](assets/tree.png)


Листья слева направо:

| № | Путь | Класс |
|---|------|-------|
| 1 | `math ≥ 80`, кружок **не** информатика/спорт, `age ≥ 16` | 1 |
| 2 | `math ≥ 80`, кружок **не** информатика/спорт, `age < 16` | 0 |
| 3 | `math ≥ 80`, кружок информатика **или** спорт | 1 |
| 4 | `math < 80`, кружок спорт, `age ≥ 14` | 1 |
| 5 | `math < 80`, кружок спорт, `age < 14` | 0 |
| 6 | `math < 80`, кружок **не** спорт | 0 |

Замечание по узлу 2: «ДА» уходит **вправо** в класс 1, «НЕТ» — влево в узел 3. Поэтому лист «класс 1 после узла 2» — это **третий** слева, а не второй.


### Решение


In [ ]:
import numpy as np
import pandas as pd


def apply_tree(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет leaf (1..6) и y (0/1) по дереву из условия."""
    math = df["math"]
    age = df["age"]
    club = df["club"].astype(str).str.strip()

    high = math >= 80
    inf_or_sport = club.isin(["информатика", "спорт"])
    sport = club.eq("спорт")

    leaf = np.empty(len(df), dtype=int)
    y = np.empty(len(df), dtype=int)

    m_left_high = high & ~inf_or_sport & (age >= 16)  # лист 1
    m_left_low = high & ~inf_or_sport & (age < 16)  # лист 2
    m_right_high = high & inf_or_sport  # лист 3
    m_sport_old = ~high & sport & (age >= 14)  # лист 4
    m_sport_young = ~high & sport & (age < 14)  # лист 5
    m_not_sport = ~high & ~sport  # лист 6

    leaf[m_left_high] = 1
    y[m_left_high] = 1
    leaf[m_left_low] = 2
    y[m_left_low] = 0
    leaf[m_right_high] = 3
    y[m_right_high] = 1
    leaf[m_sport_old] = 4
    y[m_sport_old] = 1
    leaf[m_sport_young] = 5
    y[m_sport_young] = 0
    leaf[m_not_sport] = 6
    y[m_not_sport] = 0

    out = df.copy()
    out["leaf"] = leaf
    out["y"] = y
    return out


df_students = pd.read_csv("data.csv")
print("строк:", len(df_students), "кружки:", df_students["club"].value_counts().to_dict())
pred = apply_tree(df_students)
display(pred.head())

print("1) строк класса 1:", int((pred["y"] == 1).sum()))
print("2) размеры листьев 1..6:")
leaf_counts = pred["leaf"].value_counts().reindex(range(1, 7), fill_value=0)
print(leaf_counts.tolist())

share = pred.groupby("club")["y"].mean().sort_values(ascending=False)
print("3) доля класса 1 по кружкам:")
print(share)
best = share[share == share.max()].index.tolist()
print("кружок(и) с максимальной долей:", best)


**Ответы задачи 2** (по `data.csv`, 1000 школьников):

1. В класс 1 попали **281** строка.
2. Размеры листьев слева направо: **37, 69, 107, 137, 84, 566** (сумма 1000).
3. Доля класса 1 максимальна у кружка **спорт** ($\approx 0.698$). Дальше информатика $\approx 0.220$, нет $\approx 0.077$, музыка $\approx 0.071$.

## Задача 3. Сессии пользователей

По логам действий нужно нарезать сессии: если между **двумя последовательными** событиями **одного** пользователя прошло **больше 30 минут**, начинается новая сессия. Найти `user_id` с максимальным числом сессий.


### Интуиция

Сессия — «заход в приложение». Человек потыкал, ушёл больше чем на полчаса, вернулся — это уже другой заход. Считаем заходы у каждого пользователя и берём рекордсмена по количеству заходов, а не по числу кликов.


### Вектор размышления

1. Файл в условии назван `timestamps.xlsx`, в папке он лежит как `задача 17.xlsx`, лист `logs`. Колонки `timestamp` и `user_id` (остальные поля для правила сессий не нужны).
2. Строки **не обязаны** быть отсортированы. Сначала группируем по пользователю, **сортируем по времени**, только потом смотрим разницы.
3. Первое действие пользователя всегда открывает сессию №1. Дальше: если $\Delta t > 30$ мин — номер сессии увеличивается.
4. Файл сохранён в *strict OOXML* (`purl.oclc.org/...`), обычный `pandas.read_excel` его не открывает. Перед чтением заменяем пространства имён на transitional.
5. В timestamps есть десятки **битых строк** вроде `2025-01-03T01:21114`. Их нельзя распарсить — отбрасываем (`errors='coerce'`). На ответ это не влияет: лидер единственный.


### Ключевая идея

После сортировки внутри пользователя сессии — это **1 + число разрывов** $\Delta t > 30$ мин. В pandas: `groupby('user_id')['ts'].diff() > Timedelta(minutes=30)`, затем `cumsum`.


### Материалы

- Правило inactivity timeout — стандарт аналитики мобильных приложений (часто 30 минут).
- `diff` в группе даёт `NaT` на первом событии; сравнение `NaT > 30min` даёт `False`, поэтому первое событие **не** создаёт лишний разрыв — удобно.
- Если бы в условии было «30 минут или больше», писали бы `>=`; здесь **«более 30 минут»** $\Rightarrow$ строго `>`.


### Решение


In [ ]:
from io import BytesIO
from pathlib import Path
import zipfile

import pandas as pd


def load_strict_xlsx(path: Path, usecols=None, sheet_name=0):
    """Читает xlsx в strict OOXML, который ломает openpyxl/pandas."""
    old_new = [
        (
            b"http://purl.oclc.org/ooxml/spreadsheetml/main",
            b"http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        ),
        (
            b"http://purl.oclc.org/ooxml/officeDocument/relationships",
            b"http://schemas.openxmlformats.org/officeDocument/2006/relationships",
        ),
        (
            b"http://purl.oclc.org/ooxml/officeDocument/extendedProperties",
            b"http://schemas.openxmlformats.org/officeDocument/2006/extended-properties",
        ),
        (
            b"http://purl.oclc.org/ooxml/officeDocument/docPropsVTypes",
            b"http://schemas.openxmlformats.org/officeDocument/2006/docPropsVTypes",
        ),
    ]
    buf = BytesIO()
    with zipfile.ZipFile(path) as zin, zipfile.ZipFile(
        buf, "w", compression=zipfile.ZIP_DEFLATED
    ) as zout:
        for item in zin.infolist():
            data = zin.read(item.filename)
            if item.filename.endswith(".xml"):
                for old, new in old_new:
                    data = data.replace(old, new)
            zout.writestr(item, data)
    buf.seek(0)
    return pd.read_excel(buf, sheet_name=sheet_name, usecols=usecols)


logs_path = Path("задача 17.xlsx")
logs = load_strict_xlsx(logs_path, usecols=["timestamp", "user_id"], sheet_name="logs")
logs["user_id"] = logs["user_id"].astype(str)
logs["ts"] = pd.to_datetime(logs["timestamp"], errors="coerce")

n_bad = int(logs["ts"].isna().sum())
print(f"строк: {len(logs):,}; пользователей: {logs['user_id'].nunique():,}; битых timestamp: {n_bad}")

clean = logs.dropna(subset=["ts"]).sort_values(["user_id", "ts"])
gap = pd.Timedelta(minutes=30)
is_new = clean.groupby("user_id", sort=False)["ts"].diff() > gap
clean["session"] = is_new.groupby(clean["user_id"], sort=False).cumsum() + 1

n_sessions = clean.groupby("user_id")["session"].max()
answer_uid = n_sessions.idxmax()
print("максимум сессий:", int(n_sessions.max()))
print("user_id:", answer_uid)
print("\nтоп-10:")
print(n_sessions.sort_values(ascending=False).head(10))


**Ответ задачи 3:** `user_id = 44686` (8 сессий). Следующие идут с 7 сессиями, так что ничья не возникает.


## Задача 4. Потерянная точка

Дан набор точек двух классов на плоскости и 1-NN с евклидовой метрикой. Одна точка потеряна. Известно:

- после потери 1-NN **неправильно классифицирует ровно 9 точек**;
- у потерянной точки $x + y = 16$.

Нужно вернуть координаты $(x_0, y_0)$.


### Интуиция

При $k = 1$ метка точки — это метка **ближайшего соседа**. Точку нельзя спрашивать саму у себя (иначе ошибка на обучающей выборке всегда 0). Поэтому «правильно / неправильно» здесь — это **leave-one-out**: для каждой точки ищем ближайшую **среди остальных** и сравниваем метки.

Потеря точки меняет соседей у тех, для кого она была ближайшей. Мы перебираем, *какая* точка пропала, и смотрим, у скольких оставшихся LOO-предсказание стало неверным.


### Вектор размышления

1. Сверить список с рисунком: тёмные = класс 0, светлые = класс 1.
2. Ни одна из **напечатанных** точек не имеет $x+y=16$. Значит, либо потерянная точка — это **ещё одна** точка, которой нет в списке (список = то, что осталось), либо условие «сумма 16» отбирает кандидата на целочисленной решётке.
3. Практический план, который закрывает оба чтения условия:
   - **A.** список = полный исходный набор: удаляем по одной точке, считаем LOO-ошибку на оставшихся 19;
   - **B.** список = набор *после* потери: добавляем кандидата $(x, 16-x)$ с меткой 0 или 1, считаем LOO-ошибку на 21 точке.
4. Связей по расстоянию почти нет (исключение — $(17,8)$ равноудалён от $(15,8)$ и $(19,8)$, оба класса 0, на метку не влияет).
5. Искомая точка с суммой 16, которая «логична» на рисунке, — дырка $(8, 8)$ в тёмном облаке около $x=8$.


### Ключевая идея

1-NN + LOO сводится к **графу ближайших соседей**. Достаточно полного перебора: точек мало ($\approx 20$), кандидатов с $x+y=16$ на разумной решётке — тоже десятки. Считаем ошибку напрямую, без подбора гиперпараметров.


### Материалы

Евклидово расстояние:
$$
d\bigl((x_1,y_1),(x_2,y_2)\bigr)=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}.
$$
Для сравнения соседей достаточно **квадрата** расстояния — корень монотонен.

Leave-one-out для 1-NN: $\hat y_i = y_{j^*(i)}$, где $j^*(i)=\arg\min_{j\neq i} d(i,j)$. Ошибка — число $i$ с $\hat y_i \neq y_i$.

Рисунок из условия:


![Точки kNN](assets/knn_points.png)


### Решение


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

class0 = np.array(
    [
        (1, 4),
        (1, 2),
        (2, 6),
        (4, 8),
        (8, 7),
        (8, 5),
        (8, 4),
        (15, 8),
        (15, 5),
        (15, 12),
        (17, 8),
        (19, 8),
    ],
    dtype=float,
)
class1 = np.array(
    [
        (6, 4),
        (12, 6),
        (12, 8),
        (15, 4),
        (15, 14),
        (19, 14),
        (19, 12),
        (19, 5),
    ],
    dtype=float,
)

xy = np.vstack([class0, class1])
y = np.array([0] * len(class0) + [1] * len(class1))


def loo_1nn(xy, y):
    """Предсказания и число ошибок 1-NN leave-one-out."""
    diff = xy[:, None, :] - xy[None, :, :]
    d2 = (diff ** 2).sum(axis=2)
    np.fill_diagonal(d2, np.inf)
    nn = np.argmin(d2, axis=1)
    pred = y[nn]
    return pred, int((pred != y).sum())


pred_full, err_full = loo_1nn(xy, y)
print("LOO-ошибка на полном списке:", err_full)
print("неверно классифицируются:")
print(
    pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "true": y, "pred": pred_full}).query(
        "true != pred"
    )
)

print("\nсуммы координат точек из условия:")
print([(int(a), int(b), int(a + b), int(lab)) for (a, b), lab in zip(xy, y)])
print("есть ли x+y=16 среди списка:", bool(np.any(xy.sum(axis=1) == 16)))


In [ ]:
# Сценарий A: удаляем по одной точке из списка
rows = []
for i in range(len(xy)):
    mask = np.ones(len(xy), dtype=bool)
    mask[i] = False
    _, err = loo_1nn(xy[mask], y[mask])
    rows.append(
        {
            "removed": (int(xy[i, 0]), int(xy[i, 1])),
            "class": int(y[i]),
            "x+y": int(xy[i].sum()),
            "loo_err_on_19": err,
        }
    )
tab_a = pd.DataFrame(rows).sort_values("loo_err_on_19", ascending=False)
print("Сценарий A (удаление из списка): максимум ошибки", tab_a["loo_err_on_19"].max(), "(девятки нет)")
display(tab_a.head(8))


In [ ]:
# Сценарий B: список — то, что осталось; ищем потерянную (x, 16-x)
records = []
for x in range(-2, 22):
    pt = np.array([x, 16 - x], dtype=float)
    for lab in (0, 1):
        xy2 = np.vstack([xy, pt])
        y2 = np.append(y, lab)
        _, err = loo_1nn(xy2, y2)
        records.append({"x": x, "y": 16 - x, "class": lab, "loo_err_on_21": err})

tab_b = pd.DataFrame(records).sort_values("loo_err_on_21", ascending=False)
print("Сценарий B: ближайшие к 9 ошибкам конфигурации")
display(tab_b.head(10))
print("есть ровно 9?", bool((tab_b["loo_err_on_21"] == 9).any()))


In [ ]:
# Визуально: (8, 8) — пустая клетка в тёмной группе, сумма 16
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(class0[:, 0], class0[:, 1], c="#4b2e83", s=80, label="класс 0")
ax.scatter(class1[:, 0], class1[:, 1], c="#e6c200", s=80, edgecolors="k", label="класс 1")
ax.scatter([8], [8], marker="X", s=180, c="red", label="кандидат (8, 8)")
ax.set_xticks(range(0, 21, 1))
ax.set_yticks(range(0, 16, 1))
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title("Кандидат потерянной точки: (8, 8)")
plt.show()

for lab in (0, 1):
    xy2 = np.vstack([xy, [8.0, 8.0]])
    y2 = np.append(y, lab)
    pred, err = loo_1nn(xy2, y2)
    extra_pred = pred[-1]
    extra_wrong = int(extra_pred != lab)
    print(
        f"(8, 8) как класс {lab}: LOO-ошибка всех точек = {err}; "
        f"ошибка самой точки = {extra_wrong}; ошибки среди исходных = {err - extra_wrong}"
    )


Что получается.

- Среди напечатанных точек **нет** пары с суммой 16, и удаление любой из них даёт LOO-ошибку 4…7, но не 9. Чтение «потеряли одну из списка» **не стыкуется** с обеими подсказками сразу.
- Если список — остаток после потери, а пропавшая точка лежит на прямой $x+y=16$, ровно 9 LOO-ошибок на целочисленной решётке тоже нет: максимум 8, и его дают три кандидата — $(8,8)$ класса 1, $(9,7)$ класса 1 и $(11,5)$ класса 0.
- На рисунке естественная «дырка» с суммой 16 — **$(8, 8)$**. Для светлой метки это точка-выброс в тёмном кластере: сама она классифицируется как 0, и даёт максимальную LOO-ошибку среди кандидатов на прямой $x+y=16$.

**Ответ, который стыкует рисунок и сумму 16:** $(x_0, y_0) = (8, 8)$. Условие «ровно 9 ошибок» при чистом 1-NN LOO на данном списке не воспроизводится — если проверяющая система ждёт другую точку, ориентир — таблица сценария B.


## Краткие ответы

| Задача | Ответ |
|--------|--------|
| 1 | $m=0$ → `inf`; $\sigma=0$ → `0.000000`; иначе $m$-й по величине $\lvert z_i\rvert$ (формула $\sigma$ с делением на $n$) |
| 2 | класс 1: **281**; листья 1–6: **37, 69, 107, 137, 84, 566**; максимальная доля класса 1: **спорт** |
| 3 | `user_id = 44686` |
| 4 | $(8, 8)$ |
